# marag-precision — Kaggle launcher

Thin wrapper around `src/runner.py`. **No logic lives here** — if you find yourself
editing pipeline code in this notebook, put it in the repo instead (SPEC §9).

## Before you run anything

| Setting | Value | Why |
|---|---|---|
| **Internet** | **On** (Settings → Internet) | HuggingFace model + dataset downloads fail silently without it |
| **Accelerator** | **GPU T4 x2** | Only one GPU is used; the second idles but the quota is identical |
| Model cache | `/kaggle/tmp` (set below) | `/kaggle/working` holds only ~20 GB and must not fill with weights |

Results are appended to `results/<run_id>.jsonl` **as each batch completes**, not at
the end. A session that dies at hour 11 loses nothing — rerun the same command and
it resumes (SPEC §6). Download the zip from cell 5 before the session expires.

Run IDs (SPEC §2): `baseline`, `planner_4bit`, `stepdef_4bit`, `extractor_4bit`, `qa_4bit`.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets pyyaml

In [ ]:
import os

# Keep weights off /kaggle/working (~20 GB budget, and it is what gets zipped).
os.environ['HF_HOME'] = '/kaggle/tmp/hf'
os.makedirs('/kaggle/tmp/hf', exist_ok=True)

REPO_URL = 'https://github.com/REPLACE_ME/marag-precision.git'  # <-- set this
REPO_DIR = '/kaggle/working/marag-precision'

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — set Accelerator to GPU T4 x2'
print(torch.cuda.get_device_name(0))

### Verify on 10 questions before any full run

SPEC §11 step 7. Do not launch a 300-question sweep until this cell has completed
end to end. `--seed 1234` is the development split; the real runs use the config's
`eval_seed`, which prompt work has never seen.

In [ ]:
!python smoke_test.py --run baseline --n 10 --seed 1234

In [ ]:
# The full sweep. Rerunning any line resumes it; completed calls are skipped.
RUN_ID = 'baseline'

!python -m src.runner --config config/experiment.yaml --run {RUN_ID}

In [ ]:
# Zip results for download. Safe to run repeatedly, including mid-sweep.
!cd {REPO_DIR} && zip -qr /kaggle/working/results.zip results/
!ls -lh /kaggle/working/results.zip
!ls -l {REPO_DIR}/results/